# Aquatic Center Chatbot — Tester Notebook

This notebook lets you try an experimental chatbot for an aquatic center.

The chatbot can answer questions and manage simple tasks about:

- opening hours;
- prices and subscriptions;
- facility rules;
- course booking, modification, and cancellation;
- spa booking, modification, and cancellation;
- equipment purchases;
- lost item reports.

## Evaluation form

After testing the chatbot, please complete this short evaluation form:

[Open the chatbot evaluation form](https://docs.google.com/forms/d/e/1FAIpQLSeSEDjuJKas5JmLmbZrcqRtXxvVmxRlYhRB3t5CHrq9nZVJXA/viewform?usp=publish-editor)

The form is anonymous and takes less than 2 minutes. Please submit it after completing at least one realistic interaction with the bot.

## Before running

1. Open **Runtime > Change runtime type**.
2. Select **T4 GPU** as hardware accelerator.
3. Click **Save**.
4. Run the cells from top to bottom.

A Hugging Face account or token is **not required** to run this notebook with the default public model, **Qwen3-4B-Instruct**.

The first startup can take a few minutes because the project dependencies and the model may need to be downloaded.


## Suggested things to try

The chatbot uses a mock database for testing purposes. There is no real login or authentication system: users are identified only by their name and surname inside the conversation.

The mock database already contains two example users with saved data. You can use them to test realistic scenarios such as modifying or cancelling existing bookings. You can also use any new name and surname to create new bookings or reports during the test.

You do not need to follow fixed message templates. Try to write naturally, as if you were chatting with a real assistant. The chatbot may ask follow-up questions before completing the task.

---

## Saved mock users

### Mario Rossi

Mario Rossi already has these saved course bookings:

| Activity          | Target age | Level        | Day       |
| ----------------- | ---------- | ------------ | --------- |
| `swimming_school` | adult      | intermediate | Tuesday   |
| `aquagym`         | adult      | beginner     | Wednesday |
| `hydrobike`       | adult      | intermediate | Thursday  |

Mario Rossi also has these saved lost item reports:

| Item      | Color | Location      | Date       |
| --------- | ----- | ------------- | ---------- |
| `goggles` | red   | swimming pool | 2026-02-22 |
| `towel`   | blue  | changing room | 2026-01-15 |

Things you can try with Mario Rossi:

| Scenario            | Idea                                                                                         |
| ------------------- | -------------------------------------------------------------------------------------------- |
| Cancel a course     | Try cancelling one of Mario Rossi’s existing courses                                        |
| Modify a course day | Try moving the hydrobike course to another available day                                    |
| Report a lost item  | Try reporting an item that Mario Rossi has already reported                                 |
| Test ambiguity      | Try changing an intermediate course to advanced without immediately specifying which course |

---

### Luigi Verdi

Luigi Verdi already has these saved spa bookings:

| Date       | Time  | People |
| ---------- | ----- | ------ |
| 2026-07-15 | 15:30 | 2      |
| 2026-08-22 | 16:00 | 4      |

Things you can try with Luigi Verdi:

| Scenario             | Idea                                                                         |
| -------------------- | ---------------------------------------------------------------------------- |
| Modify a spa booking | Try moving the spa booking on July 15th to another date or time             |
| Cancel a spa booking | Try cancelling one of Luigi Verdi’s existing spa bookings                   |
| Test ambiguity       | Try asking to cancel a spa booking without immediately specifying which one |

---

## General things to try

You can also try general questions or create new bookings using any name and surname.

| Area               | Idea                                                                                      |
| ------------------ | ----------------------------------------------------------------------------------------- |
| Opening hours      | Ask about the opening hours of a facility on a specific day                              |
| Prices             | Ask about the price of a subscription or ticket                                          |
| Rules              | Ask whether something is allowed in a specific area of the center                        |
| Courses            | Try booking a new course                                                                 |
| Spa                | Try booking a spa appointment                                                            |
| Shop               | Try buying an item                                                                       |
| Report lost item   | Try reporting something you lost                                                         |
| Multi-task request | Try asking two things in the same message, such as a price and an opening-hours question |


## 1. Clone the project repository

This cell downloads the latest project code from GitHub into the Colab runtime.

The local runtime folder is recreated every time this cell is executed, so the notebook uses the current repository version instead of an old shared Drive folder.


In [ ]:
from pathlib import Path
import os
import sys
import shutil
import subprocess

REPO_URL = "https://github.com/MessaAlberto/HMD_aquatic_center_chatbot.git"
PROJECT_DIR = Path("/content/HMD-project")

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

subprocess.run(
    ["git", "clone", REPO_URL, str(PROJECT_DIR)],
    check=True,
)

COLAB_UTILS_DIR = PROJECT_DIR / "colab_utils"
REQUIREMENTS_PATH = COLAB_UTILS_DIR / "requirements_colab.txt"

os.chdir(PROJECT_DIR)

for path in [PROJECT_DIR, COLAB_UTILS_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

os.environ["REQUIREMENTS_PATH"] = str(REQUIREMENTS_PATH)

print("Project folder:", PROJECT_DIR)
print("Current directory:", os.getcwd())
print("Requirements file:", REQUIREMENTS_PATH)


## 2. Check GPU

This project is designed to run with a GPU. If this cell fails, change the runtime type to **T4 GPU**, restart the session, and run the notebook again.


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. Go to Runtime > Change runtime type > T4 GPU, then restart the session.")

print("GPU:", torch.cuda.get_device_name(0))

## 3. Install requirements

This cell installs the Python packages required by the chatbot from the cloned repository.


In [ ]:
print("Installing requirements from:", REQUIREMENTS_PATH)
%pip install -q -r "$REQUIREMENTS_PATH"


## 4. Load the chatbot

This cell configures the project and loads the default model.

The default model can be changed in `MODEL_NAME`, but testers should usually keep the default value.


In [ ]:
import os
import sys
import warnings
import logging
import importlib

os.environ["APP_DEBUG"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore::FutureWarning"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=FutureWarning, module=r"bitsandbytes.*")
logging.basicConfig(level=logging.WARNING, format="%(levelname)s:%(message)s", force=True)

os.chdir(PROJECT_DIR)

for path in [PROJECT_DIR, COLAB_UTILS_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

importlib.invalidate_caches()

from colab_launcher import setup_project, load_bot

MODEL_NAME = "qwen3_4b"

setup_project(str(PROJECT_DIR), debug=False)
bot = load_bot(MODEL_NAME)

print("Chatbot loaded successfully.")

## 5. Start the chat

Run the cell below to start the chatbot.

Useful chat commands:

| Command       | Meaning                                         |
| ------------- | ----------------------------------------------- |
| `reset_state` | Reset only the current conversation state       |
| `reset`       | Reset both conversation state and mock database |
| `exit`        | Stop the chat                                   |


In [ ]:
from colab_launcher import start_chat

start_chat(bot)